# DuckDB tutorial for RAIS and firm x layer data

This notebook is a directed tutorial. The goal is to learn a small number of DuckDB and SQL patterns while building the worker-to-layer-to-firm dataset you need for the `layer_connectivity` project.

We will:
1. Open RAIS worker data in DuckDB.
2. Inspect the columns we need.
3. Standardize occupation codes as 6-character strings.
4. Reproduce the Sara layer definition from the `.do` files.
5. Collapse worker records to `firm x layer x year`.
6. Optionally fill in zero-employment cells and save the result.


## 0. Setup

DuckDB lets us query Parquet files directly with SQL. A useful workflow is:

- use `CREATE VIEW` so files behave like tables
- use `DESCRIBE` and `LIMIT` to inspect the data
- keep most work in SQL
- only convert to pandas with `.df()` when you want to display a small result


In [1]:
import duckdb
from pathlib import Path

ROOT = Path('/kellogg/proj/lgg3230/UnionSpill')
FIRM_LVL = ROOT / 'Data' / 'CBA_RAIS_firm_level'
OUT = ROOT / 'Data' / 'layer_connectivity'

WORKER_FILE = FIRM_LVL / 'worker_panel_lagos.parquet'
OUTCOME_FILE = OUT / 'firm_layer_outcomes_sara_tutorial.parquet'
OUTCOME_FULL_FILE = OUT / 'firm_layer_outcomes_sara_tutorial_full.parquet'

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='32GB'")

print('DuckDB ready')
print(WORKER_FILE)


DuckDB ready
/kellogg/proj/lgg3230/UnionSpill/Data/CBA_RAIS_firm_level/worker_panel_lagos.parquet


## 1. Read the worker panel

First we create a view over the worker panel. This does not copy the file into memory. It just gives us a table-like name, `workers`.


In [2]:
con.execute(f"""
    CREATE OR REPLACE VIEW workers AS
    SELECT *
    FROM read_parquet('{WORKER_FILE}')
""")

con.sql("DESCRIBE workers").df()


,column_name,column_type,null,key,default,extra
0,identificad,VARCHAR,YES,None,None,None
1,PIS,VARCHAR,YES,None,None,None
2,horascontr,TINYINT,YES,None,None,None
3,remdezr,DOUBLE,YES,None,None,None
4,empem3112,TINYINT,YES,None,None,None
5,tempempr,DOUBLE,YES,None,None,None
6,genero,TINYINT,YES,None,None,None
7,raca_cor,TINYINT,YES,None,None,None
8,nacionalidad,TINYINT,YES,None,None,None
9,portdefic,TINYINT,YES,None,None,None


In [3]:
con.sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT PIS) AS n_workers,
        COUNT(DISTINCT identificad) AS n_firms,
        MIN(year) AS year_min,
        MAX(year) AS year_max
    FROM workers
""").df()


,n_rows,n_workers,n_firms,year_min,year_max
0,19598473,5723160,16472,2009,2016


## 2. Inspect the key variables

For the layer construction we mainly need:

- `identificad`: firm identifier
- `PIS`: worker identifier
- `year`
- `ocup2002`: occupation code
- `remdezr`, `lr_remdezr`: wage variables
- `horascontr`: contracted hours

A good habit in SQL is to inspect a narrow slice before writing the full transformation.


In [4]:
con.sql("""
    SELECT
        identificad,
        PIS,
        year,
        ocup2002,
        remdezr,
        lr_remdezr,
        horascontr
    FROM workers
    LIMIT 10
""").df()


,identificad,PIS,year,ocup2002,remdezr,lr_remdezr,horascontr
0,00009638000355,12096588512,2009,410105,6179.78,9.127138,44
1,00009638000355,12178714562,2009,821215,1469.75,7.690948,44
2,00009638000355,12284888870,2009,911305,2655.74,8.282579,44
3,00009638000355,12411049333,2009,722110,2514.11,8.227774,44
4,00009638000355,12909416250,2009,784205,1288.39,7.559249,44
5,00009638000436,12766306260,2009,721430,2506.44,8.224719,44
6,00009638000436,20490756942,2009,721430,1504.70,7.714449,44
7,00012377000160,12342374994,2009,954120,1840.83,7.916072,44
8,00012377000160,12690325316,2009,515215,3187.32,8.465036,44
9,00012377000160,12923989319,2009,841460,1304.78,7.571890,44


## 3. Standardize `ocup2002` before slicing it

The Sara `.do` files use string logic on occupation codes. One subtle issue is that some occupation codes appear with only 5 digits. These should be treated as 6-digit codes with a leading zero.

In DuckDB, the clean fix is:

`LPAD(CAST(ocup2002 AS VARCHAR), 6, '0')`

That way, substring operations behave consistently for both 5-digit and 6-digit codes.


In [5]:
con.sql("""
    SELECT
        ocup2002,
        LPAD(CAST(ocup2002 AS VARCHAR), 6, '0') AS cbo_str,
        SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 1) AS d1,
        SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 2) AS d2,
        SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 3, 1) AS d3
    FROM workers
    WHERE ocup2002 IS NOT NULL
    ORDER BY ocup2002
    LIMIT 20
""").df()


,ocup2002,cbo_str,d1,d2,d3
0,0,000000,0,00,0
1,0,000000,0,00,0
2,0,000000,0,00,0
3,0,000000,0,00,0
4,0,000000,0,00,0
5,0,000000,0,00,0
6,0,000000,0,00,0
7,0,000000,0,00,0
8,0,000000,0,00,0
9,0,000000,0,00,0


## 4. Reproduce the Sara raw layer definition

From `sara_files/000_clean_rais.do`, the raw 6-layer mapping is:

- raw layer 1: first two digits are `11`, `12`, or `13`
- raw layer 2: first two digits are `14`
- raw layer 3: first digit is `2`
- raw layer 4: first digit is `4` to `9` and third digit is `0`
- raw layer 5: first digit is `3`
- raw layer 6: first digit is `4` to `9` and not already assigned to raw layer 4

We will build this first as a worker-level view.


In [6]:
con.execute("""
    CREATE OR REPLACE VIEW workers_sara_raw AS
    SELECT
        *,
        LPAD(CAST(ocup2002 AS VARCHAR), 6, '0') AS cbo_str,
        CASE
            WHEN SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 2) IN ('11', '12', '13') THEN 1
            WHEN SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 2) = '14' THEN 2
            WHEN SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 1) = '2' THEN 3
            WHEN SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 1) = '3' THEN 5
            WHEN SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 1) IN ('4','5','6','7','8','9')
                 AND SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 3, 1) = '0' THEN 4
            WHEN SUBSTR(LPAD(CAST(ocup2002 AS VARCHAR), 6, '0'), 1, 1) IN ('4','5','6','7','8','9') THEN 6
            ELSE NULL
        END AS layer_raw
    FROM workers
""")

con.sql("""
    SELECT *,
    FROM workers_sara_raw
    WHERE ocup2002 IS NOT NULL
    LIMIT 20
""").df()


,identificad,PIS,horascontr,remdezr,empem3112,tempempr,genero,raca_cor,nacionalidad,portdefic,...,d_apprentice,d_fixed_term,d_public,d_other_contract,ocup4,d_male,hours_ge40,d_disabled,cbo_str,layer_raw
0,05917639000150,16178296194,44,820.69,1,1.6,1,2,10,0,...,0,0,0,0,5132,1,1,0,513205,6
1,05917639000150,20459073081,44,864.77,1,9.9,0,2,10,0,...,0,0,0,0,4221,0,1,0,422125,6
2,05928246000141,10651857144,44,2922.48,1,31.4,1,8,10,0,...,0,0,0,0,8114,1,1,0,811425,6
3,05928246000141,12746995249,44,2009.37,1,31.8,1,2,10,0,...,0,0,0,0,5174,1,1,0,517420,6
4,05928246000141,12796601015,44,855.12,1,7.2,1,2,10,0,...,0,0,0,0,6210,1,1,0,621005,6
5,05928246000141,12870515148,44,4720.22,1,40.9,1,2,10,0,...,0,0,0,0,3516,1,1,0,351605,5
6,05928246000141,16195557561,44,2166.38,1,31.4,1,8,10,0,...,0,0,0,0,6410,1,1,0,641015,6
7,05928246000141,19028614179,44,1987.95,1,44.0,1,2,10,0,...,0,0,0,0,4110,1,1,0,411010,6
8,05928246000141,20047382273,44,2199.74,1,30.9,1,8,10,0,...,0,0,0,0,8413,1,1,0,841305,6
9,05932790000167,12403195431,44,1386.94,1,2.4,0,2,10,0,...,0,0,0,0,7632,0,1,0,763210,6


In [7]:
con.sql("""
    SELECT
        year,
        layer_raw,
        COUNT(*) AS n_workers
    FROM workers_sara_raw
    GROUP BY year, layer_raw
    ORDER BY year, layer_raw
""").df()


,year,layer_raw,n_workers
0,2009,1,8334
1,2009,2,39511
2,2009,3,195079
3,2009,4,70098
4,2009,5,244416
5,2009,6,1733351
6,2009,<NA>,52
7,2010,1,8742
8,2010,2,42283
9,2010,3,209494


In [8]:
con.sql("""
SELECT year,
        COUNT(DISTINCT identificad) AS n_firms
    FROM workers_sara_raw
    GROUP BY year
    ORDER BY year
""").df()

,year,n_firms
0,2009,16472
1,2010,16472
2,2011,16472
3,2012,16472
4,2013,16472
5,2014,16472
6,2015,16472
7,2016,16472


In [36]:
con.sql("""
    SELECT year, layer_raw,
        COUNT(DISTINCT PIS) AS n_workers_layers
        FROM workers_sara_raw
        GROUP BY year, layer_raw
        ORDER BY year, layer_raw
        """).df()

,year,layer_raw,n_workers_layers
0,2009,1,8183
1,2009,2,39372
2,2009,3,192866
3,2009,4,70003
4,2009,5,243405
5,2009,6,1731382
6,2009,<NA>,52
7,2010,1,8586
8,2010,2,42140
9,2010,3,207084


In [ ]:
con.sql("""
SELECT year,
        COUNT(DISTINCT (identificad, layer_raw)) AS n_total_layers,
        FROM workers_sara_raw
        GROUP BY year
        ORDER BY year
""").df()

BinderException: Binder Error: Referenced column "treat_ultra" not found in FROM clause!
Candidate bindings: "layer_raw", "tamestab", "raca_cor", "d_other_contract", "race_group"

## 5. Collapse the 6 raw layers into the 5 analysis layers

From `sara_files/010_collapse_firm.do`, the firm-level layers are adjusted as follows:

- final layer 1 = raw layers 1 and 2 combined
- final layer 2 = raw layer 3
- final layer 3 = raw layer 4
- final layer 4 = raw layer 5
- final layer 5 = raw layer 6

It is clearer to create a new variable, `layer_id`, rather than carrying five separate dummy variables.


In [14]:
con.execute("""
    CREATE OR REPLACE VIEW workers_sara AS
    SELECT
        *,
        CASE
            WHEN layer_raw IN (1, 2) THEN 1
            WHEN layer_raw = 3 THEN 2
            WHEN layer_raw = 4 THEN 3
            WHEN layer_raw = 5 THEN 4
            WHEN layer_raw = 6 THEN 5
            ELSE NULL
        END AS layer_id
    FROM workers_sara_raw
""")

con.sql("""
    SELECT
        year,
        layer_id,
        COUNT(*) AS n_workers
    FROM workers_sara
    GROUP BY year, layer_id
    ORDER BY year, layer_id
""").df()


,year,layer_id,n_workers
0,2009,1,47845
1,2009,2,195079
2,2009,3,70098
3,2009,4,244416
4,2009,5,1733351
5,2009,<NA>,52
6,2010,1,51025
7,2010,2,209494
8,2010,3,75341
9,2010,4,262167


## 6. Build the `firm x layer x year` dataset

Now we collapse worker records to the cell you want for analysis.

For each `identificad x layer_id x year`, we compute:

- `layer_emp`: number of workers in that firm-layer-year cell
- `lr_remdezr_layer`: mean log real December wage
- `lr_remdezr_h_layer`: mean log hourly wage

This mirrors the pattern already used elsewhere in `layer_connectivity`.


In [15]:
con.execute("""
    CREATE OR REPLACE VIEW firm_layer_outcomes_sara AS
    SELECT
        CAST(identificad AS VARCHAR) AS identificad,
        year,
        CAST(layer_id AS INTEGER) AS layer_id,
        COUNT(PIS) AS layer_emp,
        LOG(COUNT(PIS)) AS l_layer_emp,
        AVG(lr_remdezr) AS lr_remdezr_layer,
        AVG(LOG(remdezr / NULLIF(horascontr, 0))) AS lr_remdezr_h_layer
    FROM workers_sara
    WHERE layer_id IS NOT NULL
      AND remdezr > 0
      AND horascontr > 0
    GROUP BY identificad, year, layer_id
""")

con.sql("SELECT * FROM firm_layer_outcomes_sara LIMIT 10").df()


,identificad,year,layer_id,layer_emp,l_layer_emp,lr_remdezr_layer,lr_remdezr_h_layer
0,00352294015819,2009,2,91,1.959041,9.529108,2.325944
1,00360305003715,2009,5,58,1.763428,8.729282,2.141065
2,00360305007036,2009,5,35,1.544068,8.578459,2.075564
3,00360305017180,2009,5,55,1.740363,8.589271,2.080259
4,00360305025280,2009,5,51,1.707570,8.549772,2.063105
5,00360305033975,2009,5,10,1.000000,8.476406,2.031243
6,00360305038691,2009,5,39,1.591065,8.608161,2.088463
7,00360305059508,2009,5,13,1.113943,8.981943,2.250794
8,00360305061839,2009,5,20,1.301030,8.516445,2.048631
9,00360305072954,2009,5,27,1.431364,8.784404,2.165004


In [ ]:
con.sql("""
    SELECT
        year,
        layer_id,
        COUNT(*) AS n_firm_layer_cells,
        AVG(layer_emp) AS avg_layer_emp,
        MIN(layer_emp) AS min_layer_emp,
        MAX(layer_emp) AS max_layer_emp
    FROM firm_layer_outcomes_sara
    GROUP BY year, layer_id
    ORDER BY year, layer_id
""").df()


## 7. Optional: create zero-employment cells

Some analyses want a balanced set of `firm x layer x year` cells, including layers that a firm does not employ in that year.

The logic is:

- get all `(firm, year)` pairs that appear in the worker data
- cross them with the five possible layers
- left join the positive-employment outcomes
- replace missing employment with zero

This gives a fuller analysis dataset while keeping wages missing when a cell has zero workers.


In [ ]:
con.execute("""
    CREATE OR REPLACE VIEW firm_layer_outcomes_sara_full AS
    WITH firm_years AS (
        SELECT DISTINCT CAST(identificad AS VARCHAR) AS identificad, year
        FROM workers_sara
    ),
    all_layers AS (
        SELECT 1 AS layer_id UNION ALL
        SELECT 2 UNION ALL
        SELECT 3 UNION ALL
        SELECT 4 UNION ALL
        SELECT 5
    ),
    grid AS (
        SELECT fy.identificad, fy.year, al.layer_id
        FROM firm_years fy
        CROSS JOIN all_layers al
    )
    SELECT
        g.identificad,
        g.year,
        g.layer_id,
        COALESCE(f.layer_emp, 0) AS layer_emp,
        CASE WHEN COALESCE(f.layer_emp, 0) > 0 THEN LOG(COALESCE(f.layer_emp, 0)) ELSE NULL END AS l_layer_emp,
        LOG(1 + COALESCE(f.layer_emp, 0)) AS l1p_layer_emp,
        f.lr_remdezr_layer,
        f.lr_remdezr_h_layer
    FROM grid g
    LEFT JOIN firm_layer_outcomes_sara f
      ON g.identificad = f.identificad
     AND g.year = f.year
     AND g.layer_id = f.layer_id
""")

con.sql("SELECT * FROM firm_layer_outcomes_sara_full LIMIT 10").df()


## 8. Save the result

DuckDB can write query results directly to Parquet. This is often cleaner than pulling the full result into pandas first.

Run the next cell if you want to persist the outputs.


In [ ]:
con.execute(f"""
    COPY firm_layer_outcomes_sara
    TO '{OUTCOME_FILE}'
    (FORMAT PARQUET)
""")

con.execute(f"""
    COPY firm_layer_outcomes_sara_full
    TO '{OUTCOME_FULL_FILE}'
    (FORMAT PARQUET)
""")

print(OUTCOME_FILE)
print(OUTCOME_FULL_FILE)


## 9. What to remember

The reusable pattern is:

1. Create a view over the raw file.
2. Build a cleaned or standardized variable, here `cbo_str`.
3. Use a `CASE` statement to define `layer_id`.
4. `GROUP BY firm, layer, year` to collapse the worker data.
5. Use `COPY ... TO ... (FORMAT PARQUET)` to save the result.

Once this pattern feels natural, changing the layer definition becomes easy: the only part that changes is the `CASE` block.
